# 🚀 Omni-Channel B2B Customer Segmentation, CLV Forecasting & Churn Intelligence
**Author:** Amman Kumar Singh  
**LinkedIn:** [linkedin.com/in/amman-s-3a9301242](https://www.linkedin.com/in/amman-s-3a9301242)  
**Architecture:** Unsupervised K-Means Clustering + PCA | Random Forest CLV Regression | Gradient Boosting Churn Classification + Explainable AI (XAI)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import silhouette_score, mean_squared_error, r2_score, accuracy_score, roc_auc_score, classification_report

df = pd.read_csv('omnichannel_customer_intelligence_data.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## Stage 1: Unsupervised Behavioral Cohort Segmentation (K-Means & PCA)
Segmenting 2,500 enterprise and mid-market accounts using Recency, Frequency, Monetary (ARR), API velocity, and GenAI Copilot adoption.

In [2]:
cluster_features = [
    'Recency_Last_Active_Days', 'Tx_Frequency_90d', 'Current_ARR_USD',
    'Monthly_API_Calls_K', 'GenAI_Copilot_Adoption_Rate', 'Seat_License_Utilization'
]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
print(f'K-Means Silhouette Score: {silhouette_score(X_scaled, clusters):.4f}')

df.groupby('Customer_Persona_Cohort')[['Current_ARR_USD', 'Actual_12M_CLV_USD', 'GenAI_Copilot_Adoption_Rate', 'Churned']].mean()

## Stage 2: Supervised Ensemble Learning — Churn Risk & 12-Month CLV Forecasting
- **Gradient Boosting Classifier:** Predicts account attrition risk ahead of renewal.
- **Random Forest Regressor:** Forecasts 12-Month Customer Lifetime Value (CLV) and Net Revenue Expansion.

In [3]:
feature_cols = [
    'Contract_Tenure_Months', 'Recency_Last_Active_Days', 'Tx_Frequency_90d',
    'Current_ARR_USD', 'Monthly_API_Calls_K', 'GenAI_Copilot_Adoption_Rate',
    'Seat_License_Utilization', 'Support_Tickets_90d', 'NPS_Score',
    'Region', 'Industry', 'Segment_Tier', 'Customer_Persona_Cohort'
]
X = pd.get_dummies(df[feature_cols], drop_first=True)
y_churn = df['Churned']
y_clv = df['Actual_12M_CLV_USD']

X_train, X_test, yc_train, yc_test, yv_train, yv_test = train_test_split(
    X, y_churn, y_clv, test_size=0.25, random_state=42, stratify=y_churn
)

# 1. Gradient Boosting Churn Model
gb_clf = GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42)
gb_clf.fit(X_train, yc_train)
yc_pred = gb_clf.predict(X_test)
yc_prob = gb_clf.predict_proba(X_test)[:, 1]
print(f'Gradient Boosting Churn Accuracy: {accuracy_score(yc_test, yc_pred):.2%} | ROC-AUC: {roc_auc_score(yc_test, yc_prob):.4f}')

# 2. Random Forest CLV Regressor
rf_reg = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, yv_train)
yv_pred = rf_reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(yv_test, yv_pred))
print(f'Random Forest 12M CLV R2 Score:   {r2_score(yv_test, yv_pred):.4f} | RMSE: ${rmse:,.2f}')

## Stage 3: Explainable AI (XAI) — Permutation Feature Importance
Quantifying the exact marginal contribution of product telemetry and behavioral features on account retention.

In [4]:
perm = permutation_importance(gb_clf, X_test, yc_test, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': perm.importances_mean}).sort_values('Importance', ascending=False)
imp_df.head(10)